# WSI watershed connected-component audit

Run one globally normalized, watershed-resolved InstanSeg WSI pass over the validated SLIDE-0330 all-channel half crop, then measure connected components in the final cell plane. The audit is chunked and never loads a complete label plane into RAM. It uses 4-connectivity and reports associated nucleated cells, exact proxy cells, and unnucleated cells. The current resolver records the number of ambiguous parents but not their daughter-ID list, so watershed daughters remain within the associated-nucleated category.

In [1]:
from pathlib import Path
import json, re, subprocess, sys, time
import numpy as np
import pandas as pd
import tifffile
import zarr
from skimage.measure import label as label_equal_values
from tqdm.auto import tqdm

INSTANSEG_ROOT = Path('/data1/lowes/ratnayn/Codex/projects/instanseg')
CROP = Path('/data1/lowes/ratnayn/Codex/codex-scratch/mIF-pipeline/instanseg_watershed_production_smoke_all_channel_crop/SLIDE-0330/SLIDE-0330_all_channels_half_crop.ome.tif')
OUTPUT_DIR = CROP.parent / 'connectedness_audit'
RESOLVED_ZARR = OUTPUT_DIR / 'SLIDE-0330_watershed_resolved.zarr'
PER_CELL_CSV = OUTPUT_DIR / 'SLIDE-0330_connectedness_per_cell.csv'
SUMMARY_CSV = OUTPUT_DIR / 'SLIDE-0330_connectedness_summary.csv'
SUMMARY_JSON = OUTPUT_DIR / 'SLIDE-0330_connectedness_summary.json'
RUN_WSI = True
RUN_CONNECTEDNESS_AUDIT = True
REUSE_COMPLETED_WSI = True
WRITE_PER_CELL_CSV = True
AUDIT_CHUNK = 2048
MODEL_NAME = 'fluorescence_nuclei_and_cells'
PIXEL_SIZE_UM = 0.325
NORMALIZATION_PERCENTILES = (0.1, 99.9)
WSI_TILE_SIZE = 2048
WSI_OVERLAP = 80
WSI_DETECTION_SIZE = 20
WSI_BATCH_SIZE = 1
SEGMENTATION_CHANNELS = [
    'R1_DAPI', 'R4_P19_POLYRAT', 'R4_GFP_POLY_AF488',
    'R6_CD45_CST_AF647', 'R6_PANCK_AE1_AE3_750',
    'R12_CD31_D8V9E_AF750', 'R7_NAK_ATPASE_555',
    'R8_F480_D2S9R_555', 'R9_CD68_E3O7V_488',
    'R12_CD3E_E4T1B_AF555',
]
REFERENCE_CHANNEL = 'R1_DAPI'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if str(INSTANSEG_ROOT) not in sys.path:
    sys.path.insert(0, str(INSTANSEG_ROOT))
print({'crop': str(CROP), 'resolved_zarr': str(RESOLVED_ZARR), 'run_wsi': RUN_WSI, 'run_audit': RUN_CONNECTEDNESS_AUDIT})

{'crop': '/data1/lowes/ratnayn/Codex/codex-scratch/mIF-pipeline/instanseg_watershed_production_smoke_all_channel_crop/SLIDE-0330/SLIDE-0330_all_channels_half_crop.ome.tif', 'resolved_zarr': '/data1/lowes/ratnayn/Codex/codex-scratch/mIF-pipeline/instanseg_watershed_production_smoke_all_channel_crop/SLIDE-0330/connectedness_audit/SLIDE-0330_watershed_resolved.zarr', 'run_wsi': True, 'run_audit': True}


In [2]:
if not CROP.is_file():
    raise FileNotFoundError(CROP)
import instanseg
from instanseg import InstanSeg
from tiffslide import TiffSlide
import instanseg.inference_class as inference_class
inference_class.TiffSlide = TiffSlide
fork_path = Path(instanseg.__file__).resolve()
fork_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=INSTANSEG_ROOT, text=True).strip()
fork_status = subprocess.check_output(['git', 'status', '--porcelain'], cwd=INSTANSEG_ROOT, text=True).strip()
assert fork_path.is_relative_to(INSTANSEG_ROOT), fork_path
assert hasattr(InstanSeg, 'eval_whole_slide_image_global_normalization')
with tifffile.TiffFile(CROP) as tif:
    series = tif.series[0]
    crop_shape, crop_axes = tuple(int(v) for v in series.shape), series.axes
    ome_xml = tif.ome_metadata or ''
channel_names = [m.group(1) for m in re.finditer(r'<(?:[^:>]+:)?Channel\b[^>]*?Name=\"([^\"]*)\"', ome_xml)]
if crop_axes != 'CYX' or len(channel_names) != crop_shape[0]:
    raise ValueError(f'Expected named CYX input, got axes={crop_axes}, shape={crop_shape}, names={len(channel_names)}')
if len(channel_names) != len(set(channel_names)):
    raise ValueError('OME channel names are not unique.')
lookup = {name: index for index, name in enumerate(channel_names)}
missing = [name for name in SEGMENTATION_CHANNELS if name not in lookup]
if missing:
    raise KeyError(f'Missing segmentation channels: {missing}')
CHANNEL_IDS = [lookup[name] for name in SEGMENTATION_CHANNELS]
REFERENCE_CHANNEL_ID = lookup[REFERENCE_CHANNEL]
assert CHANNEL_IDS[0] == REFERENCE_CHANNEL_ID
print({'python': sys.executable, 'instanseg': str(fork_path), 'commit': fork_commit, 'dirty': bool(fork_status), 'crop_shape': crop_shape, 'channel_ids': CHANNEL_IDS})

{'python': '/data1/lowes/ratnayn/conda_envs/instanseg_training/bin/python', 'instanseg': '/data1/lowes/ratnayn/Codex/projects/instanseg/instanseg/__init__.py', 'commit': '8f75880b40ac3b521aecbf29a2c5ee30f386bd2a', 'dirty': False, 'crop_shape': (72, 27680, 31344), 'channel_ids': [4, 38, 42, 51, 52, 17, 56, 62, 71, 15]}


In [ ]:
def compatible_completed_output(path):
    if not path.is_dir():
        return False
    try:
        arr = zarr.open(str(path), mode='r')
        attrs = dict(arr.attrs)
        settings = attrs.get('wsi_settings') or {}
        normalization = attrs.get('normalization') or {}
        resolution = attrs.get('resolution') or {}
        return (
            attrs.get('status') == 'complete'
            and list(attrs.get('planes', [])) == ['nuclei', 'cells']
            and Path(attrs.get('source_image', '')).resolve() == CROP.resolve()
            and list(attrs.get('channel_ids', [])) == CHANNEL_IDS
            and settings.get('tile_size') == WSI_TILE_SIZE
            and settings.get('overlap') == WSI_OVERLAP
            and settings.get('detection_size') == WSI_DETECTION_SIZE
            and settings.get('resolve_cell_and_nucleus') is True
            and settings.get('resolution_method') == 'watershed'
            and [float(v) for v in normalization.get('percentiles', [])] == list(NORMALIZATION_PERCENTILES)
            and resolution.get('method') == 'watershed'
            and resolution.get('allow_unnucleated_cells') is True
        )
    except Exception:
        return False

if compatible_completed_output(RESOLVED_ZARR) and REUSE_COMPLETED_WSI:
    print('Reusing compatible completed WSI:', RESOLVED_ZARR)
elif RUN_WSI:
    if RESOLVED_ZARR.exists():
        raise ValueError(f'Existing output is incompatible; move it or choose a new path: {RESOLVED_ZARR}')
    model = InstanSeg(MODEL_NAME, verbosity=1)
    started = time.perf_counter()
    observed = model.eval_whole_slide_image_global_normalization(
        str(CROP), channel_ids=CHANNEL_IDS, pixel_size=PIXEL_SIZE_UM,
        normalization_percentiles=NORMALIZATION_PERCENTILES,
        reference_channel_id=REFERENCE_CHANNEL_ID, tile_size=WSI_TILE_SIZE,
        overlap=WSI_OVERLAP, detection_size=WSI_DETECTION_SIZE,
        batch_size=WSI_BATCH_SIZE, output_path=RESOLVED_ZARR, overwrite=False,
        resolve_cell_and_nucleus=True, resolution_method='watershed',
        allow_unnucleated_cells=True, cleanup_fragments=True, seed_threshold=0.6,
    )
    assert Path(observed).resolve() == RESOLVED_ZARR.resolve()
    print(f'WSI inference and watershed completed in {(time.perf_counter()-started)/60:.1f} min')
else:
    raise RuntimeError('No compatible result exists; set RUN_WSI=True.')
assert compatible_completed_output(RESOLVED_ZARR)

Model fluorescence_nuclei_and_cells version 0.1.1 already downloaded in /data1/lowes/ratnayn/Codex/projects/instanseg/instanseg/utils/../bioimageio_models/, loading
Requesting default device: cuda


Slide progress:   1%|          | 1/110 [00:04<08:08,  4.48s/it]/data1/lowes/ratnayn/Codex/projects/instanseg/instanseg/utils/pytorch_utils.py:312: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:49.)
  intersection = torch.sparse.mm(onehot1, onehot2.T).to_dense()
Slide progress: 100%|██████████| 110/110 [07:42<00:00,  4.21s/it]


Applying global watershed cell/nucleus resolution and validation...


In [ ]:
def connected_component_table(resolved, chunk_size=2048):
    cells, nuclei = resolved[1], resolved[0]
    height, width = map(int, cells.shape)
    parent, area, nuclear_pixels, label_id = [], [], [], []

    def find(node):
        while parent[node] != node:
            parent[node] = parent[parent[node]]
            node = parent[node]
        return node

    def union(first, second):
        root_a, root_b = find(int(first)), find(int(second))
        if root_a == root_b:
            return
        if label_id[root_a] != label_id[root_b]:
            raise RuntimeError('Attempted to union different cell IDs.')
        if area[root_a] < area[root_b]:
            root_a, root_b = root_b, root_a
        parent[root_b] = root_a
        area[root_a] += area[root_b]
        nuclear_pixels[root_a] += nuclear_pixels[root_b]

    def union_boundary(old_labels, old_nodes, new_labels, new_nodes):
        mask = (old_labels > 0) & (old_labels == new_labels)
        if not np.any(mask):
            return
        pairs = np.unique(np.stack((old_nodes[mask], new_nodes[mask]), axis=1), axis=0)
        for first, second in pairs:
            union(first, second)

    x_starts = list(range(0, width, chunk_size))
    y_starts = list(range(0, height, chunk_size))
    upper_labels, upper_nodes = [None] * len(x_starts), [None] * len(x_starts)
    for y0 in tqdm(y_starts, desc='Connectedness rows'):
        y1 = min(y0 + chunk_size, height)
        left_labels = left_nodes = None
        for column, x0 in enumerate(x_starts):
            x1 = min(x0 + chunk_size, width)
            cell_chunk = np.asarray(cells[y0:y1, x0:x1], dtype=np.int32)
            nucleus_chunk = np.asarray(nuclei[y0:y1, x0:x1], dtype=np.int32)
            components = label_equal_values(cell_chunk, background=0, connectivity=1)
            count = int(components.max())
            if count:
                component_ids, first_indices, counts = np.unique(components, return_index=True, return_counts=True)
                keep = component_ids > 0
                component_ids, first_indices, counts = component_ids[keep], first_indices[keep], counts[keep]
                component_labels = cell_chunk.ravel()[first_indices].astype(np.int64)
                if np.any(component_labels <= 0):
                    raise RuntimeError('Foreground component received a background label.')
                overlap = ((cell_chunk == nucleus_chunk) & (cell_chunk > 0)).ravel()
                component_nuclear = np.bincount(components.ravel(), weights=overlap, minlength=count + 1)[1:].astype(np.int64)
                offset = len(parent)
                for local_index in range(count):
                    node = offset + local_index
                    parent.append(node)
                    area.append(int(counts[local_index]))
                    nuclear_pixels.append(int(component_nuclear[local_index]))
                    label_id.append(int(component_labels[local_index]))
                node_map = np.where(components > 0, components.astype(np.int64) - 1 + offset, -1)
            else:
                node_map = np.full(components.shape, -1, dtype=np.int64)
            if left_labels is not None:
                union_boundary(left_labels, left_nodes, cell_chunk[:, 0], node_map[:, 0])
            if upper_labels[column] is not None:
                union_boundary(upper_labels[column], upper_nodes[column], cell_chunk[0, :], node_map[0, :])
            left_labels, left_nodes = cell_chunk[:, -1].copy(), node_map[:, -1].copy()
            upper_labels[column], upper_nodes[column] = cell_chunk[-1, :].copy(), node_map[-1, :].copy()

    roots = [index for index in range(len(parent)) if find(index) == index]
    return pd.DataFrame({
        'cell_id': [label_id[index] for index in roots],
        'component_pixels': [area[index] for index in roots],
        'nuclear_pixels': [nuclear_pixels[index] for index in roots],
    })

In [ ]:
if not RUN_CONNECTEDNESS_AUDIT:
    raise RuntimeError('Set RUN_CONNECTEDNESS_AUDIT=True to calculate metrics.')
resolved = zarr.open(str(RESOLVED_ZARR), mode='r')
attrs = dict(resolved.attrs)
validation = attrs.get('validation') or {}
assert attrs.get('status') == 'complete' and validation.get('nuclear_cell_ids_agree') is True
started = time.perf_counter()
components = connected_component_table(resolved, AUDIT_CHUNK)
components['touches_assigned_nucleus'] = components['nuclear_pixels'] > 0
components['pixels_without_nucleus_contact'] = np.where(components['touches_assigned_nucleus'], 0, components['component_pixels'])
per_cell = components.groupby('cell_id', sort=True).agg(
    component_count=('cell_id', 'size'),
    total_pixels=('component_pixels', 'sum'),
    largest_component_pixels=('component_pixels', 'max'),
    nucleus_touching_components=('touches_assigned_nucleus', 'sum'),
    pixels_outside_nucleus_components=('pixels_without_nucleus_contact', 'sum'),
).reset_index()
per_cell['secondary_component_pixels'] = per_cell['total_pixels'] - per_cell['largest_component_pixels']
per_cell['secondary_component_fraction'] = per_cell['secondary_component_pixels'] / per_cell['total_pixels']
per_cell['is_disconnected'] = per_cell['component_count'] > 1
nuclear_max, cell_max = [int(v) for v in attrs['max_label_by_plane']]
proxy_count = int(validation.get('proxy_cells', 0))
proxy_start = nuclear_max - proxy_count + 1
per_cell['category'] = np.select(
    [per_cell['cell_id'] > nuclear_max, per_cell['cell_id'] >= proxy_start],
    ['unnucleated', 'proxy'], default='associated_nucleated',
)
per_cell.loc[per_cell['category'] == 'unnucleated', 'pixels_outside_nucleus_components'] = np.nan
category_summary = per_cell.groupby('category', sort=False).agg(
    cells=('cell_id', 'size'),
    disconnected_cells=('is_disconnected', 'sum'),
    total_pixels=('total_pixels', 'sum'),
    secondary_component_pixels=('secondary_component_pixels', 'sum'),
    pixels_outside_nucleus_components=('pixels_outside_nucleus_components', 'sum'),
).reset_index()
category_summary['disconnected_fraction'] = category_summary['disconnected_cells'] / category_summary['cells']
category_summary['secondary_pixel_fraction'] = category_summary['secondary_component_pixels'] / category_summary['total_pixels']
overall = {
    'cells': int(len(per_cell)),
    'components': int(len(components)),
    'disconnected_cells': int(per_cell['is_disconnected'].sum()),
    'disconnected_fraction': float(per_cell['is_disconnected'].mean()),
    'secondary_component_pixels': int(per_cell['secondary_component_pixels'].sum()),
    'secondary_pixel_fraction': float(per_cell['secondary_component_pixels'].sum() / per_cell['total_pixels'].sum()),
    'maximum_components_for_one_cell': int(per_cell['component_count'].max()),
    'resolver_ambiguous_parent_count': int(validation.get('ambiguous_cells', 0)),
    'resolver_proxy_count': proxy_count,
    'resolver_unseeded_parent_pixels_preserved': int(validation.get('unseeded_parent_pixels_preserved', 0)),
    'audit_connectivity': 4,
    'elapsed_minutes': float((time.perf_counter() - started) / 60),
}
if WRITE_PER_CELL_CSV:
    per_cell.to_csv(PER_CELL_CSV, index=False)
category_summary.to_csv(SUMMARY_CSV, index=False)
SUMMARY_JSON.write_text(json.dumps({'overall': overall, 'categories': category_summary.to_dict(orient='records')}, indent=2) + '\n')
display(pd.DataFrame([overall]))
display(category_summary)
display(per_cell.loc[per_cell['is_disconnected']].sort_values(['secondary_component_pixels', 'component_count'], ascending=False).head(25))
print({'per_cell_csv': str(PER_CELL_CSV) if WRITE_PER_CELL_CSV else None, 'summary_csv': str(SUMMARY_CSV), 'summary_json': str(SUMMARY_JSON)})